# 09 · Análisis SQL: preguntas de negocio sobre la capa Gold

Este notebook responde **6 preguntas de negocio** con SQL sobre las tablas Gold de la plataforma. Cada consulta usa **CTE** (`WITH … AS`) y **funciones de ventana** (`OVER`, `LAG`, `RANK`, `DENSE_RANK`, `ROW_NUMBER`, promedios móviles con `ROWS BETWEEN`).

**Cómo usarlo:** conecta el notebook a **Serverless** (el mismo cómputo de los demás notebooks del proyecto) y presiona **Run all**. Cada consulta muestra su tabla de resultados debajo, como en SQL Server Management Studio. Las mismas consultas están como archivos sueltos en [`sql/analytics/`](../../sql/analytics).

| # | Pregunta | Tabla Gold | Funciones |
|---|---|---|---|
| 1 | ¿Cómo se movió el dólar cada mes de 2026? | `exchange_rate_metrics` | CTE, `LAG`, `RANK` |
| 2 | ¿Qué combustibles subieron más en 12 semanas? | `fuel_price_metrics` | CTE, `LAG(…, 12)`, `SUM … ROWS BETWEEN`, `RANK` vs `DENSE_RANK` vs `ROW_NUMBER` |
| 3 | ¿Qué grupos de gasto tuvieron más inflación cada año? | `inflation_metrics` | CTE, `LAG`, `ROW_NUMBER`, `DENSE_RANK` con `PARTITION BY` |
| 4 | ¿La inflación y el dólar se mueven juntos? | `inflation_metrics` + `exchange_rate_metrics` | 4 CTE encadenadas, `JOIN`, `AVG … ROWS BETWEEN`, `QUALIFY` |
| 5 | ¿Cuánto cuesta la canasta por quintil de ingreso? | `cost_of_living_metrics` | CTE, `LAG`, `MIN() OVER ()`, `RANK` |
| 6 | ¿Qué productos agrícolas subieron más en cada rubro? | `agri_price_monthly` | CTE, `DENSE_RANK`, `AVG/COUNT … OVER (PARTITION BY)` |

## Guía rápida de las funciones

| Función | Qué hace | Ejemplo en lenguaje simple |
|---|---|---|
| **CTE** (`WITH nombre AS (…)`) | Crea una tabla temporal con nombre que solo existe durante la consulta. Permite resolver un problema en pasos. | "Paso 1: saco la tasa diaria. Paso 2: la promedio por mes. Paso 3: comparo con el mes anterior." |
| **`OVER (…)`** | Convierte una función en función de ventana: calcula mirando otras filas **sin agrupar** ni perder el detalle. | Ver el precio de cada semana y, al lado, el promedio de las últimas 4 semanas. |
| **`PARTITION BY`** | Divide las filas en grupos; la ventana se reinicia en cada grupo. | Calcular por separado para cada combustible. |
| **`ORDER BY`** (dentro de `OVER`) | Define el orden dentro de cada grupo: qué fila es la anterior y cuál la siguiente. | Ordenar las semanas de la más vieja a la más nueva. |
| **`LAG(col, n)`** | Trae el valor de `n` filas atrás (por defecto 1). | Precio de la semana pasada, o índice de hace 12 meses. |
| **`LEAD(col, n)`** | Lo contrario: trae el valor de `n` filas adelante. | Precio de la semana siguiente. |
| **`ROW_NUMBER()`** | Numera 1, 2, 3… sin repetir, aunque haya empates. | Quedarse con el registro más reciente de cada grupo. |
| **`RANK()`** | Ranking que **repite** el puesto en empates y **salta** el siguiente: 1, 1, 3. | Dos combustibles empatados en el 1.er lugar; el siguiente queda 3.º. |
| **`DENSE_RANK()`** | Ranking que repite en empates **sin saltar**: 1, 1, 2. | Top 3 por categoría sin perder puestos. |
| **`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`** | Limita la ventana a un rango de filas: aquí, la actual y las 2 anteriores. | Promedio móvil de 3 meses. |
| **`QUALIFY`** | Filtra por el resultado de una función de ventana (como un `WHERE` para ventanas). | Quedarse solo con los últimos 24 meses. |

La celda siguiente muestra la diferencia entre `ROW_NUMBER`, `RANK` y `DENSE_RANK` con 5 filas de ejemplo.

In [0]:
%sql
-- Ejemplo: tres formas de numerar cuando hay empates
WITH notas AS (
    SELECT * FROM VALUES ('Ana', 95), ('Luis', 90), ('Marta', 90), ('Pedro', 85), ('Rosa', 80) AS t(estudiante, nota)
)
SELECT estudiante, nota,
       ROW_NUMBER() OVER (ORDER BY nota DESC) AS row_number,   -- 1,2,3,4,5  (nunca repite)
       RANK()       OVER (ORDER BY nota DESC) AS rank,         -- 1,2,2,4,5  (repite y salta)
       DENSE_RANK() OVER (ORDER BY nota DESC) AS dense_rank    -- 1,2,2,3,4  (repite sin saltar)
FROM notas
ORDER BY nota DESC;

## Explorar las tablas antes de consultar

Antes de responder cualquier pregunta conviene ver **qué tablas hay, qué columnas tienen y cómo se ven sus datos**. Estas tres celdas hacen eso.

In [0]:
%sql
SHOW TABLES IN rd_economic_intelligence.gold;

In [0]:
%sql
DESCRIBE TABLE rd_economic_intelligence.gold.fuel_price_metrics;

In [0]:
%sql
SELECT fuel_name, period_start_date, price_dop, prev_week_price_dop, weekly_change_pct, ma4_price_dop
FROM rd_economic_intelligence.gold.fuel_price_metrics
WHERE fuel_code = 'gasolina_premium'
ORDER BY period_start_date DESC
LIMIT 10;

## Pregunta 1 · ¿Cómo se movió el dólar cada mes de 2026 y cuándo se depreció más el peso?

**Por qué importa:** para una institución financiera, la tasa del dólar afecta préstamos y depósitos en moneda extranjera, y el poder de compra de los clientes.

**Paso a paso**
1. **CTE `tasa_diaria`:** toma la tasa de **venta** USD/DOP de cada día.
2. **CTE `promedio_mensual`:** agrupa por mes (`date_trunc('MONTH', …)`) y calcula el promedio y los días hábiles.
3. **CTE `con_mes_anterior`:** con `LAG(tasa_promedio) OVER (ORDER BY mes)` pone al lado de cada mes el promedio del mes anterior.
4. **Consulta final:** calcula la variación % y la ordena con `RANK()`: el puesto 1 es el mes en que más subió el dólar, o sea, en que más se depreció el peso.

**Cómo leer el resultado:** `variacion_pct` positiva = el dólar subió ese mes; negativa = el peso se apreció.

**Hallazgo (corrida del 19-nov-2026):** de enero a agosto el peso se apreció casi todos los meses (el dólar bajó de RD$63.51 a RD$58.68). En **octubre** el dólar subió **3.17 %**, la mayor depreciación del año, y en septiembre 1.11 %.

In [0]:
%sql
-- P1. ¿Cómo se movió el dólar cada mes de 2026 y cuál fue el mes de mayor depreciación del peso?
WITH tasa_diaria AS (                       -- Paso 1: tasa de venta diaria del dólar
    SELECT rate_date, rate
    FROM rd_economic_intelligence.gold.exchange_rate_metrics
    WHERE rate_type = 'sell' AND currency_pair = 'USD/DOP'
),
promedio_mensual AS (                       -- Paso 2: promedio de cada mes
    SELECT date_trunc('MONTH', rate_date) AS mes,
           ROUND(AVG(rate), 4)            AS tasa_promedio,
           COUNT(*)                       AS dias_habiles
    FROM tasa_diaria
    GROUP BY date_trunc('MONTH', rate_date)
),
con_mes_anterior AS (                       -- Paso 3: traer el mes anterior con LAG
    SELECT mes, tasa_promedio, dias_habiles,
           LAG(tasa_promedio) OVER (ORDER BY mes) AS tasa_mes_anterior
    FROM promedio_mensual
)
SELECT date_format(mes, 'yyyy-MM')                                        AS mes,
       tasa_promedio,
       tasa_mes_anterior,
       ROUND((tasa_promedio / tasa_mes_anterior - 1) * 100, 2)            AS variacion_pct,
       RANK() OVER (ORDER BY tasa_promedio / tasa_mes_anterior DESC)     AS ranking_depreciacion
FROM con_mes_anterior
WHERE year(mes) = 2026
ORDER BY mes;

## Pregunta 2 · ¿Qué combustibles subieron más en las últimas 12 semanas, y cuántas semanas subieron?

**Por qué importa:** el combustible pega en transporte y en el costo de casi todo. Un alza sostenida presiona la inflación y la capacidad de pago de los clientes.

**Paso a paso**
1. **CTE `principales`:** filtra los 8 combustibles de uso común.
2. **CTE `con_hace_12_semanas`:**
   - `LAG(price_dop, 12) OVER (PARTITION BY fuel_name ORDER BY period_start_date)` trae el precio de **12 semanas atrás** de cada combustible.
   - `CASE WHEN price_dop > LAG(price_dop) …` marca con 1 cada semana en que el precio subió respecto a la anterior.
   - `ROW_NUMBER() … ORDER BY period_start_date DESC` numera las semanas de la más reciente a la más vieja.
3. **CTE `con_conteo`:** `SUM(subio_esta_semana) OVER (… ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)` cuenta en cuántas de las **últimas 12 semanas** subió el precio (suma móvil).
4. **CTE `ultima_semana`:** se queda con `orden_reciente = 1`, la semana más reciente de cada combustible, y calcula el cambio en RD$ y en %.
5. **Consulta final:** ordena con las tres funciones de ranking sobre `semanas_con_subida` para ver la diferencia con datos reales.

**Cómo leer el resultado:** AVTUR y Kerosene subieron **9 de 12 semanas** y empatan.
- `RANK` les da 1 y 1, y el siguiente queda **3** (salta el 2).
- `DENSE_RANK` les da 1 y 1, y el siguiente queda **2** (no salta).
- `ROW_NUMBER` no permite empates: 1 y 2 (desempata por el % de alza).

**Hallazgo:** AVTUR (+41.2 %) y Kerosene (+40.8 %) son los que más subieron en 12 semanas; el GLP es el único que bajó (−1.5 %). *Nota: la última semana publicada por el MICM en esta tabla es la del 25-sep-2026.*

In [0]:
%sql
-- P2. ¿Qué combustibles subieron más en las últimas 12 semanas y cuántas semanas subieron? (RANK vs DENSE_RANK vs ROW_NUMBER)
WITH principales AS (                        -- Paso 1: los combustibles que usa la gente
    SELECT fuel_name, period_start_date, price_dop
    FROM rd_economic_intelligence.gold.fuel_price_metrics
    WHERE fuel_code IN ('gasolina_premium','gasolina_regular','gasoil_optimo',
                        'gasoil_regular','glp','kerosene','avtur','fuel_oil')
),
con_hace_12_semanas AS (                     -- Paso 2: precio de 12 semanas atrás con LAG(…, 12)
    SELECT fuel_name, period_start_date, price_dop,
           LAG(price_dop, 12) OVER (PARTITION BY fuel_name ORDER BY period_start_date) AS precio_hace_12_sem,
           CASE WHEN price_dop > LAG(price_dop) OVER (PARTITION BY fuel_name ORDER BY period_start_date)
                THEN 1 ELSE 0 END                                                          AS subio_esta_semana,
           ROW_NUMBER()       OVER (PARTITION BY fuel_name ORDER BY period_start_date DESC) AS orden_reciente
    FROM principales
),
con_conteo AS (                              -- Paso 3: cuántas de las últimas 12 semanas subió (suma móvil)
    SELECT *,
           SUM(subio_esta_semana) OVER (PARTITION BY fuel_name ORDER BY period_start_date
                                        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS semanas_con_subida
    FROM con_hace_12_semanas
),
ultima_semana AS (                           -- Paso 4: quedarnos con la semana más reciente
    SELECT fuel_name, period_start_date AS semana, price_dop AS precio_actual, precio_hace_12_sem, semanas_con_subida,
           ROUND(price_dop - precio_hace_12_sem, 2)                        AS cambio_dop,
           ROUND((price_dop / precio_hace_12_sem - 1) * 100, 2)            AS cambio_pct
    FROM con_conteo
    WHERE orden_reciente = 1
)
SELECT fuel_name AS combustible, semana, precio_actual, precio_hace_12_sem, cambio_dop, cambio_pct,
       RANK() OVER (ORDER BY cambio_pct DESC)                AS puesto_por_alza,
       semanas_con_subida,
       RANK()       OVER (ORDER BY semanas_con_subida DESC)   AS rank_subidas,
       DENSE_RANK() OVER (ORDER BY semanas_con_subida DESC)   AS dense_rank_subidas,
       ROW_NUMBER() OVER (ORDER BY semanas_con_subida DESC, cambio_pct DESC) AS row_number_subidas
FROM ultima_semana
ORDER BY semanas_con_subida DESC, cambio_pct DESC;

## Pregunta 3 · ¿Qué grupos de gasto tuvieron la inflación interanual más alta cada año?

**Por qué importa:** muestra en qué rubros se encareció más la vida de los hogares (comida, transporte, vivienda…), útil para entender la presión sobre el ingreso disponible.

**Paso a paso**
1. **CTE `grupos`:** índice mensual (IPC) de los 12 grupos de gasto.
2. **CTE `interanual`:** `LAG(index_value, 12) OVER (PARTITION BY grupo ORDER BY reference_month)` trae el índice de **hace 12 meses** de cada grupo.
3. **CTE `cierre_anual`:** calcula la inflación interanual `(índice / índice_hace_12m − 1) × 100` y, con `ROW_NUMBER() … ORDER BY reference_month DESC` dentro de cada grupo y año, marca el **último mes de cada año** (diciembre, o el último disponible en 2026).
4. **CTE `ranking`:** `DENSE_RANK() OVER (PARTITION BY anio ORDER BY inflación DESC)` ordena los grupos **dentro de cada año**.
5. **Consulta final:** se queda con el **top 3** de cada año desde 2021.

**Hallazgo:** en 2026 (a octubre) lidera **Transporte (9.30 %)**; en 2025 fue **Alimentos y Bebidas (8.19 %)**; en 2021, Transporte llegó a **16.24 %**. *Restaurantes y Hoteles* aparece en el top 3 los seis años, de 2021 a 2026.

In [0]:
%sql
-- P3. ¿Qué grupos de gasto tuvieron la inflación interanual más alta cada año? (Top 3 por año)
WITH grupos AS (                              -- Paso 1: índice mensual de cada grupo de gasto
    SELECT series_name AS grupo, reference_month, index_value
    FROM rd_economic_intelligence.gold.inflation_metrics
    WHERE series_type = 'expenditure_group'
),
interanual AS (                               -- Paso 2: índice de hace 12 meses con LAG
    SELECT grupo, reference_month, index_value,
           LAG(index_value, 12) OVER (PARTITION BY grupo ORDER BY reference_month) AS indice_hace_12m
    FROM grupos
),
cierre_anual AS (                             -- Paso 3: inflación interanual de diciembre (o el último mes del año en curso)
    SELECT grupo, year(reference_month) AS anio, reference_month,
           ROUND((index_value / indice_hace_12m - 1) * 100, 2) AS inflacion_interanual_pct,
           ROW_NUMBER() OVER (PARTITION BY grupo, year(reference_month) ORDER BY reference_month DESC) AS ultimo_mes
    FROM interanual
    WHERE indice_hace_12m IS NOT NULL
),
ranking AS (                                  -- Paso 4: ordenar los grupos dentro de cada año
    SELECT anio, date_format(reference_month, 'yyyy-MM') AS mes_cierre, grupo, inflacion_interanual_pct,
           DENSE_RANK() OVER (PARTITION BY anio ORDER BY inflacion_interanual_pct DESC) AS puesto
    FROM cierre_anual
    WHERE ultimo_mes = 1 AND anio >= 2021
)
SELECT * FROM ranking
WHERE puesto <= 3
ORDER BY anio DESC, puesto;

## Pregunta 4 · ¿La inflación y el dólar se mueven juntos?

**Por qué importa:** si el dólar sube, los importados se encarecen y eso suele empujar la inflación. Ver ambas series juntas ayuda a anticipar riesgos de tasas y de crédito.

**Paso a paso**
1. **CTE `ipc`:** inflación interanual nacional calculada con `LAG(index_value, 12) OVER (ORDER BY reference_month)`.
2. **CTE `dolar_mensual`:** promedio mensual de la tasa de venta.
3. **CTE `dolar_yoy`:** variación interanual del dólar, otra vez con `LAG(…, 12)`.
4. **CTE `unidos`:** `JOIN` de las dos series por mes.
5. **Consulta final:**
   - `brecha_pp` = inflación − variación del dólar, en puntos porcentuales.
   - `AVG(…) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` = **promedio móvil de 3 meses**, que suaviza los saltos de un mes.
   - `QUALIFY ROW_NUMBER() OVER (ORDER BY mes DESC) <= 24` deja solo los **últimos 24 meses**.

**Hallazgo:** desde 2026 se separan. La inflación sigue entre 4.6 % y 5.7 % mientras el dólar **baja** frente al año anterior (−5.8 % en agosto y septiembre). La brecha llega a **10.9 puntos**: la inflación de 2026 no la está empujando el tipo de cambio.

In [0]:
%sql
-- P4. ¿La inflación y el dólar se mueven juntos? Comparación interanual de los últimos 24 meses
WITH ipc AS (                                  -- Paso 1: inflación interanual nacional (calculada con LAG)
    SELECT reference_month AS mes,
           ROUND((index_value / LAG(index_value, 12) OVER (ORDER BY reference_month) - 1) * 100, 2) AS inflacion_yoy_pct
    FROM rd_economic_intelligence.gold.inflation_metrics
    WHERE series_type = 'headline'
),
dolar_mensual AS (                             -- Paso 2: tasa de venta promedio de cada mes
    SELECT date_trunc('MONTH', rate_date) AS mes, AVG(rate) AS tasa_promedio
    FROM rd_economic_intelligence.gold.exchange_rate_metrics
    WHERE rate_type = 'sell'
    GROUP BY date_trunc('MONTH', rate_date)
),
dolar_yoy AS (                                 -- Paso 3: variación interanual del dólar con LAG
    SELECT mes, ROUND(tasa_promedio, 2) AS tasa_promedio,
           ROUND((tasa_promedio / LAG(tasa_promedio, 12) OVER (ORDER BY mes) - 1) * 100, 2) AS dolar_yoy_pct
    FROM dolar_mensual
),
unidos AS (                                    -- Paso 4: unir ambas series por mes
    SELECT i.mes, i.inflacion_yoy_pct, d.tasa_promedio, d.dolar_yoy_pct
    FROM ipc i JOIN dolar_yoy d ON i.mes = d.mes
    WHERE i.inflacion_yoy_pct IS NOT NULL
)
SELECT date_format(mes, 'yyyy-MM') AS mes,
       inflacion_yoy_pct,
       dolar_yoy_pct,
       ROUND(inflacion_yoy_pct - dolar_yoy_pct, 2)                                         AS brecha_pp,
       ROUND(AVG(inflacion_yoy_pct) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS inflacion_prom_3m,
       ROUND(AVG(dolar_yoy_pct)     OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS dolar_prom_3m
FROM unidos
QUALIFY ROW_NUMBER() OVER (ORDER BY mes DESC) <= 24
ORDER BY mes;

## Pregunta 5 · ¿Cuánto cuesta la canasta familiar por quintil de ingreso y cuánto subió en un año?

**Por qué importa:** el quintil 1 es el 20 % de hogares de menor ingreso y el quintil 5 el de mayor ingreso. Saber cuánto cuesta vivir en cada segmento ayuda a evaluar la capacidad de pago de los clientes.

**Paso a paso**
1. **CTE `quintiles`:** costo mensual de la canasta de cada quintil.
2. **CTE `con_anio_anterior`:** `LAG(basket_cost_dop, 12) OVER (PARTITION BY quintil ORDER BY reference_month)` trae el costo de hace 12 meses.
3. **CTE `ultimo_mes`:** se queda con el mes más reciente usando una subconsulta `MAX(reference_month)`.
4. **Consulta final:**
   - `aumento_dop` y `aumento_pct` frente al año anterior.
   - `basket_cost_dop / MIN(basket_cost_dop) OVER ()`: `OVER ()` vacío usa **todas las filas** como ventana, así cada quintil se compara con el más barato (quintil 1).
   - `RANK()` ordena por el % de aumento.

**Hallazgo (octubre 2026):** la canasta del quintil 5 cuesta **RD$81,727.62**, **2.73 veces** la del quintil 1 (RD$29,972.34). El quintil 5 es el que más subió (**+5.30 %**) y el quintil 1 el que menos (+4.73 %).

In [0]:
%sql
-- P5. ¿Cuánto cuesta la canasta familiar por quintil de ingreso y cuánto subió en un año?
WITH quintiles AS (                            -- Paso 1: costo mensual de la canasta por quintil
    SELECT segment AS quintil, reference_month, basket_cost_dop
    FROM rd_economic_intelligence.gold.cost_of_living_metrics
    WHERE segment_type = 'income_quintile'
),
con_anio_anterior AS (                         -- Paso 2: costo de hace 12 meses con LAG
    SELECT quintil, reference_month, basket_cost_dop,
           LAG(basket_cost_dop, 12) OVER (PARTITION BY quintil ORDER BY reference_month) AS costo_hace_12m
    FROM quintiles
),
ultimo_mes AS (                                -- Paso 3: solo el mes más reciente
    SELECT * FROM con_anio_anterior
    WHERE reference_month = (SELECT MAX(reference_month) FROM quintiles)
)
SELECT quintil,
       date_format(reference_month, 'yyyy-MM')                                AS mes,
       ROUND(basket_cost_dop, 2)                                             AS costo_canasta_dop,
       ROUND(costo_hace_12m, 2)                                              AS costo_hace_12m_dop,
       ROUND(basket_cost_dop - costo_hace_12m, 2)                            AS aumento_dop,
       ROUND((basket_cost_dop / costo_hace_12m - 1) * 100, 2)                AS aumento_pct,
       ROUND(basket_cost_dop / MIN(basket_cost_dop) OVER (), 2)              AS veces_quintil_1,
       RANK() OVER (ORDER BY basket_cost_dop / costo_hace_12m DESC)          AS ranking_aumento
FROM ultimo_mes
ORDER BY quintil;

## Pregunta 6 · ¿Qué productos agrícolas subieron más en el último año dentro de cada rubro?

**Por qué importa:** los alimentos pesan mucho en el gasto de los hogares de menor ingreso. Identificar qué productos se dispararon ayuda a explicar la inflación de alimentos.

**Paso a paso**
1. **CTE `ultimo_mes`:** obtiene el mes más reciente disponible.
2. **CTE `precios`:** precio minorista promedio de todos los mercados (`market = 'Promedio mercados'`) y su alza interanual, para ese mes.
3. **CTE `ranking`:**
   - `DENSE_RANK() OVER (PARTITION BY rubro ORDER BY yoy_pct DESC)` ordena los productos **dentro de cada rubro**.
   - `AVG(yoy_pct) OVER (PARTITION BY rubro)` pone al lado el alza promedio del rubro, **sin agrupar**: se ve el detalle del producto y el promedio de su grupo en la misma fila.
   - `COUNT(*) OVER (PARTITION BY rubro)` cuenta cuántos productos tiene el rubro.
4. **Consulta final:** top 3 de cada rubro.

**Hallazgo (octubre 2026):** el **Mango Gota de Oro** subió **56.6 %**, aunque el rubro de frutas en promedio casi no cambió (0.12 %). En hortalizas, el **repollo** subió 46.5 % y el **ajo criollo** 39.8 %. Raíces y tubérculos es el rubro con mayor alza promedio (12.79 %).

In [0]:
%sql
-- P6. ¿Qué productos agrícolas subieron más en el último año dentro de cada rubro? (Top 3, mercado minorista)
WITH ultimo_mes AS (                           -- Paso 1: el mes más reciente con datos
    SELECT MAX(reference_month) AS mes
    FROM rd_economic_intelligence.gold.agri_price_monthly
),
precios AS (                                   -- Paso 2: precio promedio de todos los mercados por producto
    SELECT a.rubro, a.product, a.unit, a.avg_price_dop, ROUND(a.yoy_pct, 2) AS yoy_pct, a.markets_quoted
    FROM rd_economic_intelligence.gold.agri_price_monthly a
    JOIN ultimo_mes u ON a.reference_month = u.mes
    WHERE a.channel = 'retail' AND a.market = 'Promedio mercados' AND a.yoy_pct IS NOT NULL
),
ranking AS (                                   -- Paso 3: ordenar por alza dentro de cada rubro
    SELECT rubro, product, unit, avg_price_dop, yoy_pct,
           DENSE_RANK() OVER (PARTITION BY rubro ORDER BY yoy_pct DESC)  AS puesto_en_rubro,
           ROUND(AVG(yoy_pct) OVER (PARTITION BY rubro), 2)            AS alza_promedio_rubro,
           COUNT(*) OVER (PARTITION BY rubro)                          AS productos_en_rubro
    FROM precios
)
SELECT rubro, puesto_en_rubro, product AS producto, unit AS unidad,
       ROUND(avg_price_dop, 2) AS precio_promedio_dop, yoy_pct AS alza_interanual_pct,
       alza_promedio_rubro, productos_en_rubro
FROM ranking
WHERE puesto_en_rubro <= 3
ORDER BY rubro, puesto_en_rubro;

## Resumen

| Pregunta | Respuesta corta (corrida del 19-nov-2026) |
|---|---|
| Dólar 2026 | El peso se apreció de enero a agosto; en octubre el dólar subió 3.17 %, la mayor alza del año. |
| Combustibles | AVTUR (+41.2 %) y Kerosene (+40.8 %) lideran en 12 semanas, con 9 semanas de subida cada uno. |
| Inflación por grupo | 2026: Transporte 9.30 %; 2025: Alimentos 8.19 %. |
| Inflación vs dólar | En 2026 la inflación (~5 %) sube mientras el dólar baja: brecha de hasta 10.9 puntos. |
| Canasta por quintil | Quintil 5 = 2.73 veces el quintil 1; todos subieron entre 4.7 % y 5.3 % en un año. |
| Agricultura | Mango (+56.6 %), repollo (+46.5 %) y ajo (+39.8 %) son las mayores alzas de su rubro. |

Los números cambian cada lunes, cuando el Job de Lakeflow actualiza las tablas Gold.